# Google Play: Pre-launch Model Training (notebook 2 of 2)
**Input:** `playstore_clean_dataset.csv` from notebook 1 (no empty cells, label already assigned).
**Question:** can a model that sees only pre-launch fields tell well-received games from not-well-received ones?

**Inputs the model sees (12):** category, content_rating, free, price_usd, ad_supported, in_app_purchases, size_mb, min_android, has_website, has_privacy, size_was_missing, min_android_was_missing.
**Never seen by the model:** rating, rating count, installs, last updated, editors choice, app name, release date, developer. The notebook asserts this.

**Evaluation:** developer-grouped 5-fold CV (a studio never appears in both train and test), then a temporal check (train on earlier releases, test on the latest 20%). Metrics: macro F1, AUC, F1 of the not-well-received class. Baseline: always predict the majority class.

In [1]:
import json, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

DATA = 'playstore_clean_dataset.csv'
RANDOM_STATE, N_SPLITS, TEMPORAL_TEST_FRACTION = 42, 5, 0.2
CAT = ['category', 'content_rating']
NUM = ['free', 'price_usd', 'ad_supported', 'in_app_purchases', 'size_mb', 'min_android',
       'has_website', 'has_privacy', 'size_was_missing', 'min_android_was_missing']
FEATURES = CAT + NUM
FORBIDDEN_EXACT = {'label', 'audit_rating', 'audit_rating_count', 'rating', 'rating_count', 'app_id', 'developer_hash', 'release_date', 'release_year'}
FORBIDDEN_PART = ['install', 'updated', 'editors', 'scraped', 'app_name']
assert not [f for f in FEATURES if f in FORBIDDEN_EXACT or any(w in f for w in FORBIDDEN_PART)], 'forbidden column among features'

df = pd.read_csv(DATA)
FEATURES = [f for f in FEATURES if f in df.columns]   # notebook 1 may have dropped constant columns
assert df[FEATURES + ['label']].isna().sum().sum() == 0, 'empty cells: rerun notebook 1'
y = df['label'].values; groups = df['developer_hash'].values
print(f'rows {len(df):,} | well received {y.mean()*100:.1f}% | studios {df["developer_hash"].nunique():,} | features {len(FEATURES)}')

rows 109,363 | well received 60.0% | studios 40,692 | features 12


In [2]:
def make_model(seed=RANDOM_STATE):
    return CatBoostClassifier(depth=4, learning_rate=0.1, iterations=500, l2_leaf_reg=5, auto_class_weights='Balanced',
                              cat_features=[c for c in CAT if c in FEATURES], random_seed=seed, verbose=0, thread_count=4)

def score(y_true, p):
    pred = (p >= 0.5).astype(int)
    return dict(macro_f1=f1_score(y_true, pred, average='macro'), auc=roc_auc_score(y_true, p),
                f1_not_well_received=f1_score(y_true, pred, pos_label=0))

def grouped_cv(cols, seed=RANDOM_STATE):
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    rows = []
    for tr, te in skf.split(df, y, groups):
        m = make_model(seed)
        m.set_params(cat_features=[c for c in CAT if c in cols])
        m.fit(df.loc[tr, cols], y[tr])
        rows.append(score(y[te], m.predict_proba(df.loc[te, cols])[:, 1]))
    r = pd.DataFrame(rows)
    return r.mean().round(4), r.std().round(4)

maj = max(y.mean(), 1 - y.mean())
print(f'majority-class accuracy: {maj:.3f} (macro F1 of always predicting one class is about 0.5 or lower)')
mean, std = grouped_cv(FEATURES)
print('\nGrouped 5-fold CV, all 12 inputs'); print(pd.DataFrame({'mean': mean, 'std': std}))

majority-class accuracy: 0.600 (macro F1 of always predicting one class is about 0.5 or lower)

Grouped 5-fold CV, all 12 inputs
                        mean     std
macro_f1              0.6085  0.0120
auc                   0.6658  0.0089
f1_not_well_received  0.5680  0.0052


## Temporal check (train on earlier releases, test on the latest 20%)

In [3]:
d = df.dropna(subset=['release_date']).sort_values('release_date')
cut = int(len(d) * (1 - TEMPORAL_TEST_FRACTION))
tr, te = d.iloc[:cut], d.iloc[cut:]
m = make_model(); m.fit(tr[FEATURES], tr['label'])
print(f'train through {tr["release_date"].iloc[-1]} ({len(tr):,}) | test {te["release_date"].iloc[0]} to {te["release_date"].iloc[-1]} ({len(te):,})')
print('well received: train', round(tr['label'].mean()*100,1), '% | test', round(te['label'].mean()*100,1), '%')
print(pd.Series(score(te['label'].values, m.predict_proba(te[FEATURES])[:, 1])).round(4))

train through 2020-03-09 (86,016) | test 2020-03-09 to 2021-06-15 (21,505)
well received: train 59.2 % | test 63.4 %
macro_f1                0.5932
auc                     0.6546
f1_not_well_received    0.5403
dtype: float64


## Tier ablation (which groups of inputs add anything)

In [4]:
TIERS = [('T1 category + content rating', ['category', 'content_rating']),
         ('T2 + price and free', ['free', 'price_usd']),
         ('T3 + ads and in-app purchases', ['ad_supported', 'in_app_purchases']),
         ('T4 + size and Android version', ['size_mb', 'min_android', 'size_was_missing', 'min_android_was_missing']),
         ('T5 + website and privacy flags', ['has_website', 'has_privacy'])]
cols, out = [], []
for name, add in TIERS:
    cols += [c for c in add if c in FEATURES]
    mean, std = grouped_cv(list(cols))
    out.append({'tier': name, 'macro_f1': mean['macro_f1'], 'auc': mean['auc'], 'f1_not_well_received': mean['f1_not_well_received']})
ablation = pd.DataFrame(out); ablation

,tier,macro_f1,auc,f1_not_well_received
0,T1 category + content rating,0.5681,0.6110,0.5298
1,T2 + price and free,0.5719,0.6170,0.5382
2,T3 + ads and in-app purchases,0.5989,0.6478,0.5579
3,T4 + size and Android version,0.6062,0.6629,0.5677
4,T5 + website and privacy flags,0.6089,0.6657,0.5695


## Permutation importance (what the final model relies on)

In [5]:
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
tr_i, te_i = next(skf.split(df, y, groups))
m = make_model(); m.fit(df.loc[tr_i, FEATURES], y[tr_i])
Xte, yte = df.loc[te_i, FEATURES].copy(), y[te_i]
base = roc_auc_score(yte, m.predict_proba(Xte)[:, 1]); rng = np.random.default_rng(0); imp = {}
for c in FEATURES:
    drops = []
    for _ in range(5):
        Xp = Xte.copy(); Xp[c] = rng.permutation(Xp[c].values)
        drops.append(base - roc_auc_score(yte, m.predict_proba(Xp)[:, 1]))
    imp[c] = round(float(np.mean(drops)), 4)
pd.Series(imp).sort_values(ascending=False).to_frame('AUC drop when shuffled')

,AUC drop when shuffled
category,0.0755
in_app_purchases,0.0270
size_mb,0.0239
min_android,0.0108
content_rating,0.0094
price_usd,0.0065
has_website,0.0059
ad_supported,0.0047
has_privacy,0.0041
size_was_missing,0.0005


## Reading the results (fill in after the run)
- If grouped-CV macro F1 is close to 0.5 and the AUC close to 0.5, pre-launch store fields carry little signal on this platform. Report that as the finding. The Steam module had a description text to lean on and this file has none.
- Compare the temporal result with grouped CV. A big drop means the grouped result was helped by period effects.
- Compare with the Steam baseline (macro F1 0.6603) only as a rough guide. The labels, populations and feature sets differ.
- Report numbers as measured. Do not tune on the temporal test set.